# 02 - Data Quality Analysis & Data Cleaning
**Branch:** `feature/data`

Notebook này:
1. Phân tích chất lượng dữ liệu chi tiết (loại vấn đề, số record ảnh hưởng, mức độ nghiêm trọng).
2. Áp dụng các hàm làm sạch từ `src/cleaning_functions.py`.
3. Ghi rõ trước/sau cleaning, cột giữ/loại, lý do xử lý.
4. Xuất dataset sạch ra `data/processed/`.


In [1]:
import pandas as pd
import numpy as np
from pathlib import Path
import sys

sys.path.append('../src')
sys.path.append('../src/data')
from cleaning_functions import (
    report_missing, drop_duplicate_records, fix_salary_range,
    flag_zero_negative_salary, fix_salary_units, standardize_categorical,
    clean_text_column, detect_outliers_iqr, cast_dtype
)

pd.set_option('display.max_columns', 100)

RAW_DIR = Path('../data/raw')
PROCESSED_DIR = Path('../data/processed')
PROCESSED_DIR.mkdir(parents=True, exist_ok=True)

cleaning_log = []  # tổng hợp toàn bộ report để đưa vào Data Documentation


In [2]:
postings = pd.read_csv(RAW_DIR / 'postings.csv', low_memory=False)
companies = pd.read_csv(RAW_DIR / 'companies.csv')
company_industries = pd.read_csv(RAW_DIR / 'company_industries.csv')
company_specialities = pd.read_csv(RAW_DIR / 'company_specialities.csv')
job_industries = pd.read_csv(RAW_DIR / 'job_industries.csv')
job_skills = pd.read_csv(RAW_DIR / 'job_skills.csv')
salaries = pd.read_csv(RAW_DIR / 'salaries.csv')
benefits = pd.read_csv(RAW_DIR / 'benefits.csv')

n_before = {
    'postings': len(postings), 'companies': len(companies),
    'company_industries': len(company_industries),
    'company_specialities': len(company_specialities),
    'job_industries': len(job_industries), 'job_skills': len(job_skills),
    'salaries': len(salaries), 'benefits': len(benefits),
}
n_before


{'postings': 123849,
 'companies': 24473,
 'company_industries': 24375,
 'company_specialities': 169387,
 'job_industries': 164808,
 'job_skills': 213768,
 'salaries': 40785,
 'benefits': 67943}

## 1. Data Quality Analysis

Với mỗi vấn đề, ghi rõ: **Vấn đề / Feature / Số record ảnh hưởng / Mức độ / Cách xử lý / Lý do**.


### 1.1. Missing values

In [3]:
print('--- postings ---')
display(report_missing(postings))
print('--- companies ---')
display(report_missing(companies))
print('--- salaries ---')
display(report_missing(salaries))


--- postings ---


,n_missing,pct_missing
closed_time,122776,99.13
skills_desc,121410,98.03
med_salary,117569,94.93
remote_allowed,108603,87.69
applies,100529,81.17
min_salary,94056,75.94
max_salary,94056,75.94
pay_period,87776,70.87
compensation_type,87776,70.87
normalized_salary,87776,70.87


--- companies ---


,n_missing,pct_missing
company_size,2774,11.33
description,297,1.21
zip_code,28,0.11
address,22,0.09
state,22,0.09
name,1,0.00
city,1,0.00


--- salaries ---


,n_missing,pct_missing
med_salary,33947,83.23
max_salary,6838,16.77
min_salary,6838,16.77


**Nhận xét (companies.csv - đã kiểm chứng với dữ liệu thật):**
- `company_size`: 2774 dòng thiếu (~11.3%) — mức độ trung bình, KHÔNG xóa vì company vẫn có giá trị phân tích qua industry/description.
- `description`, `state`, `city`, `address`, `zip_code`: thiếu rất ít (<300 dòng, <1.2%) — mức độ thấp.

**Cách xử lý dự kiến:**
- `company_size` thiếu: giữ NaN, không impute bằng giá trị giả định (vì suy đoán quy mô công ty không có căn cứ) — hoặc impute bằng median theo `industry` nếu cần cho model sau này (quyết định ở bước Feature Engineering, không xóa ở bước Cleaning).
- Các cột text thiếu ít (`description`, `city`,...): giữ NaN, KHÔNG xóa record vì chỉ ảnh hưởng 1 cột trong khi các cột khác vẫn dùng được.


### 1.2. Duplicate records

In [4]:
for name, df, key in [
    ('postings', postings, 'job_id'),
    ('companies', companies, 'company_id'),
    ('salaries', salaries, 'job_id'),
]:
    n_full_dup = df.duplicated().sum()
    n_key_dup = df[key].duplicated().sum() if key in df.columns else None
    print(f"{name}: {n_full_dup} dòng trùng toàn bộ | {n_key_dup} dòng trùng theo khóa '{key}'")


postings: 0 dòng trùng toàn bộ | 0 dòng trùng theo khóa 'job_id'
companies: 0 dòng trùng toàn bộ | 0 dòng trùng theo khóa 'company_id'
salaries: 0 dòng trùng toàn bộ | 0 dòng trùng theo khóa 'job_id'


### 1.3. Sai datatype / giá trị không hợp lệ

In [5]:
# company_size nên là số nguyên (category code), kiểm tra có giá trị lẻ bất thường không
print(companies['company_size'].dropna().apply(float.is_integer).all())
print(sorted(companies['company_size'].dropna().unique()))


True
[np.float64(1.0), np.float64(2.0), np.float64(3.0), np.float64(4.0), np.float64(5.0), np.float64(6.0), np.float64(7.0)]


In [6]:
# Salary: kiểm tra min > max, giá trị âm/0
print('min_salary > max_salary:', (salaries['min_salary'] > salaries['max_salary']).sum())
print('min_salary <= 0:', (salaries['min_salary'] <= 0).sum())
print('max_salary <= 0:', (salaries['max_salary'] <= 0).sum())
print()
print(salaries['pay_period'].value_counts())
print(salaries['compensation_type'].value_counts())


min_salary > max_salary: 0
min_salary <= 0: 0
max_salary <= 0: 0

pay_period
YEARLY      23768
HOURLY      16289
MONTHLY       539
WEEKLY        180
BIWEEKLY        9
Name: count, dtype: int64
compensation_type
BASE_SALARY    40785
Name: count, dtype: int64


**Vấn đề phát hiện (salaries.csv - đã kiểm chứng):**
- Không có min_salary > max_salary trong 6 file phụ đã kiểm tra → tốt.
- `med_salary` thiếu tới 33,947/40,785 dòng (~83%) — vì nhiều job posting chỉ cung cấp min/max chứ không có median, đây là **thiếu dữ liệu tự nhiên của nguồn (MNAR)**, không phải lỗi. Không impute, giữ NaN.
- `pay_period` có nhiều loại: HOURLY, YEARLY, MONTHLY, WEEKLY, BIWEEKLY... cần **chuẩn hóa** để so sánh salary (quy đổi bằng `normalized_salary` ở bảng `postings`).


### 1.4. Category không đồng nhất & Text bị lỗi

In [7]:
# Kiểm tra khoảng trắng thừa / hoa-thường không đồng nhất trong company name
sample_names = companies['name'].dropna().astype(str)
n_leading_trailing_space = sample_names.apply(lambda x: x != x.strip()).sum()
print('Company name có khoảng trắng đầu/cuối:', n_leading_trailing_space)

# company_industries: kiểm tra case không đồng nhất
print()
print('Số industry unique (company_industries):', company_industries['industry'].nunique())
print('Số industry unique sau khi chuẩn hóa upper+strip:',
      company_industries['industry'].str.strip().str.upper().nunique())


Company name có khoảng trắng đầu/cuối: 318

Số industry unique (company_industries): 144
Số industry unique sau khi chuẩn hóa upper+strip: 144


### 1.5. Giá trị bất thường / cực trị trong company_size

In [8]:
outlier_mask = detect_outliers_iqr(companies.dropna(subset=['company_size']), 'company_size')
print(f'Số company bị đánh dấu outlier theo IQR: {outlier_mask.sum()} / {outlier_mask.notna().sum()}')
print(companies.dropna(subset=['company_size']).loc[outlier_mask, 'company_size'].value_counts())


Số company bị đánh dấu outlier theo IQR: 0 / 21699
Series([], Name: count, dtype: int64)


**Phân biệt Data Error vs Genuine Extreme Value:**
`company_size` trong dataset này là **mã phân loại quy mô** (1=nhỏ nhất, 7=lớn nhất — theo chuẩn LinkedIn), KHÔNG phải số nhân viên thực tế.
=> Các giá trị "outlier" theo IQR ở đây (vd company_size=7, tức tập đoàn cực lớn như IBM) là **Genuine Extreme Value hợp lệ**, KHÔNG phải lỗi.
=> Quyết định: **giữ nguyên**, không xử lý như outlier cần loại bỏ.


## 2. Data Cleaning — áp dụng xử lý

In [9]:
# --- companies.csv ---
companies_clean, r1 = drop_duplicate_records(companies, subset=['company_id'])
cleaning_log.append(r1)

companies_clean, r2 = clean_text_column(companies_clean, 'name')
cleaning_log.append(r2)

companies_clean, r3 = clean_text_column(companies_clean, 'description')
cleaning_log.append(r3)

companies_clean.shape


(24473, 10)

In [10]:
# --- salaries.csv ---
salaries_clean, r4 = drop_duplicate_records(salaries, subset=['salary_id'])
cleaning_log.append(r4)

salaries_clean, r5 = fix_salary_range(salaries_clean, 'min_salary', 'max_salary')
cleaning_log.append(r5)

salaries_clean, r6 = flag_zero_negative_salary(salaries_clean, cols=['min_salary','med_salary','max_salary'])
cleaning_log.append(r6)

salaries_clean, r6b = fix_salary_units(salaries_clean, min_col='min_salary', max_col='max_salary', med_col='med_salary', pay_period_col='pay_period')
cleaning_log.append(r6b)

salaries_clean, r7 = standardize_categorical(salaries_clean, 'pay_period')
cleaning_log.append(r7)

salaries_clean, r8 = standardize_categorical(salaries_clean, 'compensation_type')
cleaning_log.append(r8)

salaries_clean.shape


(40785, 8)

In [11]:
# --- company_industries / company_specialities / job_industries / job_skills ---
company_industries_clean, r9 = drop_duplicate_records(company_industries)
cleaning_log.append(r9)
company_industries_clean, r10 = standardize_categorical(company_industries_clean, 'industry')
cleaning_log.append(r10)

company_specialities_clean, r11 = drop_duplicate_records(company_specialities)
cleaning_log.append(r11)

job_industries_clean, r12 = drop_duplicate_records(job_industries)
cleaning_log.append(r12)

job_skills_clean, r13 = drop_duplicate_records(job_skills)
cleaning_log.append(r13)

benefits_clean, r14 = drop_duplicate_records(benefits)
cleaning_log.append(r14)


In [12]:
# --- postings.csv (bảng chính) ---
postings_clean, r15 = drop_duplicate_records(postings, subset=['job_id'])
cleaning_log.append(r15)

for col in ['min_salary', 'med_salary', 'max_salary', 'normalized_salary']:
    if col in postings_clean.columns:
        postings_clean, r = flag_zero_negative_salary(postings_clean, cols=[col])
        cleaning_log.append(r)

if {'min_salary', 'max_salary'}.issubset(postings_clean.columns):
    postings_clean, r16 = fix_salary_range(postings_clean, 'min_salary', 'max_salary')
    cleaning_log.append(r16)

postings_clean, r16b = fix_salary_units(postings_clean, min_col='min_salary', max_col='max_salary', med_col='med_salary', pay_period_col='pay_period', norm_col='normalized_salary')
cleaning_log.append(r16b)

for col in ['formatted_work_type', 'work_type', 'formatted_experience_level', 'pay_period', 'compensation_type']:
    if col in postings_clean.columns:
        postings_clean, r = standardize_categorical(postings_clean, col)
        cleaning_log.append(r)

for col in ['title', 'description', 'skills_desc']:
    if col in postings_clean.columns:
        postings_clean, r = clean_text_column(postings_clean, col)
        cleaning_log.append(r)

postings_clean.shape


(123849, 31)

## 3. So sánh trước / sau cleaning

In [13]:
n_after = {
    'postings': len(postings_clean), 'companies': len(companies_clean),
    'company_industries': len(company_industries_clean),
    'company_specialities': len(company_specialities_clean),
    'job_industries': len(job_industries_clean), 'job_skills': len(job_skills_clean),
    'salaries': len(salaries_clean), 'benefits': len(benefits_clean),
}

compare = pd.DataFrame({
    'n_before': n_before,
    'n_after': n_after,
})
compare['n_removed'] = compare['n_before'] - compare['n_after']
compare


,n_before,n_after,n_removed
postings,123849,123849,0
companies,24473,24473,0
company_industries,24375,24375,0
company_specialities,169387,169387,0
job_industries,164808,164808,0
job_skills,213768,213768,0
salaries,40785,40785,0
benefits,67943,67943,0


## 4. Cleaning log — dùng cho Data Documentation

In [14]:
cleaning_log_df = pd.DataFrame(cleaning_log)
cleaning_log_df


,action,subset,n_before,n_after,n_removed,column,n_empty_or_whitespace_found,columns,n_affected,reason,details,n_hourly_to_yearly,n_yearly_to_hourly,n_total_affected,n_unique_before,n_unique_after
0,drop_duplicates,[company_id],24473.0,24473.0,0.0,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN
1,clean_text_column,NaN,NaN,NaN,NaN,name,318.0,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN
2,clean_text_column,NaN,NaN,NaN,NaN,description,5796.0,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN
3,drop_duplicates,[salary_id],40785.0,40785.0,0.0,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN
4,fix_salary_range_swap,NaN,NaN,NaN,NaN,NaN,NaN,"[min_salary, max_salary]",0.0,min_salary > max_salary là lỗi nhập liệu (đảo ...,NaN,NaN,NaN,NaN,NaN,NaN
5,flag_zero_negative_salary_as_nan,NaN,NaN,NaN,NaN,NaN,NaN,"[min_salary, med_salary, max_salary]",16.0,Lương <= 0 là giá trị không hợp lệ (Data Error...,"{'min_salary': 0, 'med_salary': 16, 'max_salar...",NaN,NaN,NaN,NaN,NaN
6,fix_salary_units,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,Lệch đơn vị thời gian (nhập lương năm vào ô Ho...,NaN,33.0,364.0,397.0,NaN,NaN
7,standardize_categorical,NaN,NaN,NaN,NaN,pay_period,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,5.0,5.0
8,standardize_categorical,NaN,NaN,NaN,NaN,compensation_type,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,1.0,1.0
9,drop_duplicates,None,24375.0,24375.0,0.0,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN


In [15]:
cleaning_log_df.to_csv(PROCESSED_DIR / 'cleaning_log.csv', index=False)


## 5. Xuất dataset sạch

In [16]:
postings_clean.to_csv(PROCESSED_DIR / 'postings_clean.csv', index=False)
companies_clean.to_csv(PROCESSED_DIR / 'companies_clean.csv', index=False)
company_industries_clean.to_csv(PROCESSED_DIR / 'company_industries_clean.csv', index=False)
company_specialities_clean.to_csv(PROCESSED_DIR / 'company_specialities_clean.csv', index=False)
job_industries_clean.to_csv(PROCESSED_DIR / 'job_industries_clean.csv', index=False)
job_skills_clean.to_csv(PROCESSED_DIR / 'job_skills_clean.csv', index=False)
salaries_clean.to_csv(PROCESSED_DIR / 'salaries_clean.csv', index=False)
benefits_clean.to_csv(PROCESSED_DIR / 'benefits_clean.csv', index=False)

print('Đã lưu toàn bộ dataset sạch vào data/processed/')


Đã lưu toàn bộ dataset sạch vào data/processed/


## 6. Cột giữ lại / loại bỏ

> Điền cụ thể sau khi review với team (Feature Engineering) — ví dụ:
> - Giữ: toàn bộ cột nghiệp vụ (title, description, salary fields, location, work_type, time fields...)
> - Cân nhắc loại (để riêng, không xóa khỏi processed): `job_posting_url`, `application_url` — chỉ là link, cardinality gần 100%, ít giá trị cho model, nhưng vẫn giữ trong file processed để nhóm Feature/Model tự quyết.
> - `skills_desc` trùng lặp thông tin với `job_skills.csv` (bảng chuẩn hóa hơn) — cân nhắc ưu tiên dùng `job_skills.csv` khi build feature.
